In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import pandas as pd
import numpy as np

In [3]:
# Load cleaned data
train = pd.read_csv('../data/train_cleaned.csv', parse_dates=['Date'])

In [4]:
# Sort by Store and Date — important for lag/rolling features
train = train.sort_values(['Store', 'Date']).reset_index(drop=True)

In [5]:
print(f"Shape: {train.shape}")
print(f"Date range: {train['Date'].min()} to {train['Date'].max()}")
train.head()

Shape: (844392, 19)
Date range: 2013-01-01 00:00:00 to 2015-07-31 00:00:00


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,Month,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,3,2013-01-02,5530,668,1,0,0,1,1,c,a,1270.0,9.0,2008.0,0,0.0,0.0,NaN
1,1,4,2013-01-03,4327,578,1,0,0,1,1,c,a,1270.0,9.0,2008.0,0,0.0,0.0,NaN
2,1,5,2013-01-04,4486,619,1,0,0,1,1,c,a,1270.0,9.0,2008.0,0,0.0,0.0,NaN
3,1,6,2013-01-05,4997,635,1,0,0,1,1,c,a,1270.0,9.0,2008.0,0,0.0,0.0,NaN
4,1,1,2013-01-07,7176,785,1,1,0,1,1,c,a,1270.0,9.0,2008.0,0,0.0,0.0,NaN


In [6]:
# Extract date features
train['Year'] = train['Date'].dt.year
train['Month'] = train['Date'].dt.month
train['Week'] = train['Date'].dt.isocalendar().week.astype(int)
train['Day'] = train['Date'].dt.day
train['IsWeekend'] = (train['DayOfWeek'] >= 6).astype(int)
train['IsMonthStart'] = train['Date'].dt.is_month_start.astype(int)
train['IsMonthEnd'] = train['Date'].dt.is_month_end.astype(int)

print("Date features added!")
print(train[['Date', 'Year', 'Month', 'Week', 'Day', 'IsWeekend', 'IsMonthStart', 'IsMonthEnd']].head(10))

Date features added!
        Date  Year  Month  Week  Day  IsWeekend  IsMonthStart  IsMonthEnd
0 2013-01-02  2013      1     1    2          0             0           0
1 2013-01-03  2013      1     1    3          0             0           0
2 2013-01-04  2013      1     1    4          0             0           0
3 2013-01-05  2013      1     1    5          1             0           0
4 2013-01-07  2013      1     2    7          0             0           0
5 2013-01-08  2013      1     2    8          0             0           0
6 2013-01-09  2013      1     2    9          0             0           0
7 2013-01-10  2013      1     2   10          0             0           0
8 2013-01-11  2013      1     2   11          0             0           0
9 2013-01-12  2013      1     2   12          1             0           0


In [7]:
# Lag features : sales from past weeks for same store
train['Sales_Lag_7'] = train.groupby('Store')['Sales'].shift(7)
train['Sales_Lag_14'] = train.groupby('Store')['Sales'].shift(14)
train['Sales_Lag_28'] = train.groupby('Store')['Sales'].shift(28)

print("Lag features added!")
print(train[['Store', 'Date', 'Sales', 'Sales_Lag_7', 'Sales_Lag_14', 'Sales_Lag_28']].head(20))

Lag features added!
    Store       Date  Sales  Sales_Lag_7  Sales_Lag_14  Sales_Lag_28
0       1 2013-01-02   5530          NaN           NaN           NaN
1       1 2013-01-03   4327          NaN           NaN           NaN
2       1 2013-01-04   4486          NaN           NaN           NaN
3       1 2013-01-05   4997          NaN           NaN           NaN
4       1 2013-01-07   7176          NaN           NaN           NaN
5       1 2013-01-08   5580          NaN           NaN           NaN
6       1 2013-01-09   5471          NaN           NaN           NaN
7       1 2013-01-10   4892       5530.0           NaN           NaN
8       1 2013-01-11   4881       4327.0           NaN           NaN
9       1 2013-01-12   4952       4486.0           NaN           NaN
10      1 2013-01-14   4717       4997.0           NaN           NaN
11      1 2013-01-15   3900       7176.0           NaN           NaN
12      1 2013-01-16   4008       5580.0           NaN           NaN
13      1 2013

In [8]:
# Rolling mean features — avg sales over past N days for same store
train['Rolling_Mean_7'] = train.groupby('Store')['Sales'].transform(
    lambda x: x.shift(1).rolling(window=7, min_periods=1).mean()
)

train['Rolling_Mean_14'] = train.groupby('Store')['Sales'].transform(
    lambda x: x.shift(1).rolling(window=14, min_periods=1).mean()
)

train['Rolling_Mean_30'] = train.groupby('Store')['Sales'].transform(
    lambda x: x.shift(1).rolling(window=30, min_periods=1).mean()
)

print("Rolling features added!")
print(train[['Store', 'Date', 'Sales', 'Rolling_Mean_7', 'Rolling_Mean_14', 'Rolling_Mean_30']].head(20))

Rolling features added!
    Store       Date  Sales  Rolling_Mean_7  Rolling_Mean_14  Rolling_Mean_30
0       1 2013-01-02   5530             NaN              NaN              NaN
1       1 2013-01-03   4327     5530.000000      5530.000000      5530.000000
2       1 2013-01-04   4486     4928.500000      4928.500000      4928.500000
3       1 2013-01-05   4997     4781.000000      4781.000000      4781.000000
4       1 2013-01-07   7176     4835.000000      4835.000000      4835.000000
5       1 2013-01-08   5580     5303.200000      5303.200000      5303.200000
6       1 2013-01-09   5471     5349.333333      5349.333333      5349.333333
7       1 2013-01-10   4892     5366.714286      5366.714286      5366.714286
8       1 2013-01-11   4881     5275.571429      5307.375000      5307.375000
9       1 2013-01-12   4952     5354.714286      5260.000000      5260.000000
10      1 2013-01-14   4717     5421.285714      5229.200000      5229.200000
11      1 2013-01-15   3900     5381.285

In [10]:
# Encode StoreType (a, b, c, d) -> numbers
train['StoreType_enc'] = train['StoreType'].map({'a': 0, 'b': 1, 'c': 2, 'd': 3})

# Encode Assortment (a, b, c) -> numbers
train['Assortment_enc'] = train['Assortment'].map({'a': 0, 'b': 1, 'c': 2})

# Encode StateHoliday (0, a, b, c) -> numbers
train['StateHoliday_enc'] = train['StateHoliday'].map({'0': 0, 'a': 1, 'b': 2, 'c': 3})

print("Encoded!")
print(train[['StoreType', 'StoreType_enc', 'Assortment', 'Assortment_enc', 'StateHoliday', 'StateHoliday_enc']].drop_duplicates().sort_values('StoreType_enc'))

Encoded!
       StoreType  StoreType_enc Assortment  Assortment_enc StateHoliday  \
401265         a              0          c               2            c   
781            a              0          a               0            0   
803            a              0          a               0            0   
1011           a              0          a               0            a   
4687           a              0          c               2            0   
4709           a              0          c               2            0   
4917           a              0          c               2            a   
91431          a              0          c               2            b   
194140         b              1          b               1            0   
63809          b              1          a               0            c   
425387         b              1          c               2            c   
63478          b              1          a               0            0   
63452          b

In [12]:
# Check final shape and null counts
print(f"Final shape: {train.shape}")
print(f"\nNull counts:")
print(train.isnull().sum()[train.isnull().sum() > 0])

# Save
train.to_csv('../data/train_features.csv', index=False)
print("\nSaved to data/train_features.csv")

Final shape: (844392, 34)

Null counts:
PromoInterval       423307
Sales_Lag_7           7805
Sales_Lag_14         15610
Sales_Lag_28         31220
Rolling_Mean_7        1115
Rolling_Mean_14       1115
Rolling_Mean_30       1115
StateHoliday_enc    294912
dtype: int64

Saved to data/train_features.csv
